# Gold-Set: Frankenstein — 20 neue Items für ein zweites Buch

**Warum?** Das Gold-Set v2 hat 36 Items aus *einem* Buch (Dracula); viele Vergleiche sind knapp
nicht signifikant. Ein zweites, gemeinfreies Buch (Mary Shelley, *Frankenstein*, Gutenberg #84)
erhöht n auf 56 und zeigt, ob Befunde über ein Buch hinaus tragen.

Gleicher Workflow und gleiche Regeln wie `gold_set_v2.ipynb`: passagen-genaue Spans, stratifizierte
Stichprobe, Fragetypen fact/paraphrase/multi/unanswerable ≈ 50/25/15/10 % (hier 10/5/3/2).
Jedes Item bekommt `group: "Frankenstein"` — die Eval-Skripte suchen damit in der richtigen Gruppe
(`--gold eval/gold_v2.jsonl,eval/gold_frankenstein.jsonl`).

**LLM-Entwürfe sind hier eingeschaltet** (Entscheidung für diesen Durchgang), aber nur als
Rohmaterial: jede Frage in eigenen Worten umformulieren und die Antwort gegen den Text prüfen —
sonst übernimmt die Frage Wortlaut aus dem Chunk und das Retrieval findet sie trivial (Leakage).

In [ ]:
import json
import random
import urllib.request

import pandas as pd

QDRANT_URL = "http://localhost:6333"
OLLAMA_URL = "http://localhost:11434"
COLLECTION = "book_chunks"
EMBED_MODEL = "bge-m3"

GROUP_ID = "Frankenstein"    # zweites Buch (gemeinfrei)
K = 10                        # Kandidaten pro Such-Vorschau
SEED = 42                     # Reproduzierbare Stichproben

GOLD_OUT = "../eval/gold_frankenstein.jsonl"   # wird nie überschrieben
ID_PREFIX = "f"

pd.set_option("display.max_colwidth", 100)


def _post(url: str, body: dict) -> dict:
    req = urllib.request.Request(
        url, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"}
    )
    return json.load(urllib.request.urlopen(req))

## 1. Chunks der Gruppe laden

In [ ]:
def load_chunks(group_id: str) -> pd.DataFrame:
    rows, offset = [], None
    while True:
        body = {
            "limit": 256, "offset": offset, "with_payload": True, "with_vector": False,
            "filter": {"must": [{"key": "group_id", "match": {"value": group_id}}]},
        }
        res = _post(f"{QDRANT_URL}/collections/{COLLECTION}/points/scroll", body)["result"]
        for p in res["points"]:
            pl = p["payload"]
            rows.append({
                "book_id": pl["book_id"], "book_title": pl.get("book_title"),
                "chapter": pl.get("chapter"), "chunk_index": pl["chunk_index"],
                "char_start": pl["char_start"], "char_end": pl["char_end"],
                "n_chars": pl["char_end"] - pl["char_start"], "text": pl["text"],
            })
        offset = res.get("next_page_offset")
        if offset is None:
            break
    return pd.DataFrame(rows).sort_values(["book_id", "chunk_index"]).reset_index(drop=True)


chunks = load_chunks(GROUP_ID)
print(f"{len(chunks)} Chunks in Gruppe '{GROUP_ID}'")
chunks[["book_title", "chapter", "chunk_index", "char_start", "char_end", "n_chars"]].head()

## 2. Stichprobe: 20 Passagen, über das Buch verteilt, mit Fragetyp-Plan

Je Drittel des Buchs werden Passagen gezogen und ein Fragetyp zugewiesen (10 fact, 5 paraphrase,
3 multi, 2 unanswerable). Für `unanswerable` brauchst du keine Passage — frag nach etwas, das
plausibel klingt, aber nicht im Buch steht (z. B. „Wie heißt Frankensteins Hund?“).

In [ ]:
random.seed(SEED)
PLAN = ["fact"] * 10 + ["paraphrase"] * 5 + ["multi"] * 3 + ["unanswerable"] * 2
random.shuffle(PLAN)

buckets = pd.qcut(chunks["chunk_index"], 3, labels=["Anfang", "Mitte", "Ende"])
needs_passage = [q for q in PLAN if q != "unanswerable"]
per_bucket = [len(needs_passage) // 3 + (1 if i < len(needs_passage) % 3 else 0) for i in range(3)]
picks = pd.concat([
    part.sample(n=n, random_state=SEED)
    for (_, part), n in zip(chunks.groupby(buckets, observed=True), per_bucket)
])
sample_rows = []
for (_, r), qtype in zip(picks.iterrows(), needs_passage):
    sample_rows.append({"qtype": qtype, "chapter": r["chapter"], "chunk_index": r["chunk_index"],
                        "char_start": r["char_start"], "char_end": r["char_end"],
                        "preview": r["text"][:200].replace("\n", " ") + " …"})
print(f"{len(sample_rows)} Passagen + {PLAN.count('unanswerable')} unanswerable")
pd.DataFrame(sample_rows)

## 3. LLM-Entwürfe (Rohmaterial, nicht übernehmen!)

Ein Entwurf je Passage. Danach: Passage im Volltext lesen (`read_chunk`), Frage in eigenen Worten
schreiben, Antwort selbst formulieren.

In [ ]:
GENERATE_DRAFTS = True  # für diesen Durchgang an; Ollama muss laufen

def draft_question(chunk_text: str, model: str = "qwen2.5:7b-instruct-q4_K_M") -> str:
    body = {
        "model": model, "stream": False,
        "messages": [{
            "role": "user",
            "content": (
                "Formuliere EINE deutsche Quizfrage zu folgendem Romanausschnitt. "
                "Die Frage darf keine Formulierungen aus dem Text übernehmen und muss "
                "ohne den Text verständlich sein. Antworte NUR mit der Frage.\n\n" + chunk_text
            ),
        }],
    }
    return _post(f"{OLLAMA_URL}/api/chat", body)["message"]["content"].strip()


if GENERATE_DRAFTS:
    for s in sample_rows:
        row = chunks[chunks["chunk_index"] == s["chunk_index"]].iloc[0]
        print(f"[{s['qtype']}] chunk {s['chunk_index']} ({s['chapter']}):")
        print("  Entwurf:", draft_question(row["text"]), "\n")
else:
    print("GENERATE_DRAFTS=False — Zelle übersprungen.")

In [ ]:
def read_chunk(chunk_index: int) -> None:
    row = chunks[chunks["chunk_index"] == chunk_index].iloc[0]
    print(f"{row['book_title']} — {row['chapter']}  (chunk {row['chunk_index']}, "
          f"Zeichen {row['char_start']}–{row['char_end']})\n")
    print(row["text"])


read_chunk(int(sample_rows[0]["chunk_index"]))

## 4. Spans prüfen: echte Pipeline-Suche + Nachbar-Chunks

Wie in v2: die Passage labeln, nicht den Chunk; bei Overlap Nachbarn prüfen.

In [ ]:
def _l2(v):
    s = sum(x * x for x in v) ** 0.5
    return [x / s for x in v] if s else v


def preview_search(question: str, k: int = K) -> pd.DataFrame:
    emb = _post(f"{OLLAMA_URL}/api/embed", {"model": EMBED_MODEL, "input": [question]})["embeddings"][0]
    body = {
        "query": _l2(emb), "limit": k, "with_payload": True,
        "filter": {"must": [{"key": "group_id", "match": {"value": GROUP_ID}}]},
    }
    pts = _post(f"{QDRANT_URL}/collections/{COLLECTION}/points/query", body)["result"]["points"]
    return pd.DataFrame([{
        "rank": i + 1, "score": round(p["score"], 3),
        "chapter": p["payload"].get("chapter"), "chunk_index": p["payload"]["chunk_index"],
        "char_start": p["payload"]["char_start"], "char_end": p["payload"]["char_end"],
        "preview": p["payload"]["text"][:160].replace("\n", " ") + " …",
    } for i, p in enumerate(pts)])


def show_neighbors(chunk_index: int, radius: int = 1) -> pd.DataFrame:
    sel = chunks[chunks["chunk_index"].between(chunk_index - radius, chunk_index + radius)]
    out = sel[["chapter", "chunk_index", "char_start", "char_end"]].copy()
    out["preview"] = sel["text"].str[:160].str.replace("\n", " ", regex=False) + " …"
    return out


preview_search("Warum flieht Victor nach der Erschaffung vor seinem Geschöpf?")

## 5. Items erfassen (`group` wird automatisch gesetzt)

In [ ]:
gold_new: list[dict] = []

_book_extent = chunks.groupby("book_id")["char_end"].max().to_dict()
_default_book = chunks["book_id"].iloc[0]


def _merge_spans(spans: list[tuple[int, int]]) -> list[dict]:
    merged: list[list[int]] = []
    for start, end in sorted(spans):
        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return [{"char_start": s, "char_end": e} for s, e in merged]


def add_item(item_id: str, question: str, answer: str, spans: list[tuple[int, int]],
             qtype: str = "fact", chapters: list[str] | None = None,
             book_id: str | None = None, question_en: str = "", note: str = "",
             source: str = "manual") -> None:
    assert qtype in {"fact", "paraphrase", "multi", "unanswerable"}, f"unbekannter qtype: {qtype}"
    assert item_id not in {g["id"] for g in gold_new}, f"ID {item_id} existiert schon"
    assert question.strip() and answer.strip(), "Frage/Antwort darf nicht leer sein"
    book = book_id or _default_book
    if qtype == "unanswerable":
        assert not spans, "unanswerable-Items haben keine Spans"
    else:
        assert spans, "mindestens ein Span noetig (oder qtype='unanswerable')"
        extent = _book_extent[book]
        for s, e in spans:
            assert 0 <= s < e <= extent, f"Span ({s},{e}) ausserhalb des Buchs (0..{extent})"
    gold_new.append({
        "id": item_id, "question": question.strip(), "question_en": question_en.strip(),
        "answer": answer.strip(), "book_id": book, "spans": _merge_spans(spans),
        "qtype": qtype, "chapters": chapters or [], "note": note, "source": source,
        "group": GROUP_ID,
    })
    print(f"ok — {len(gold_new)} Items erfasst")



# Beispiel (auskommentiert lassen, bis die Frage geprüft ist):
# add_item("f01", question="…", answer="…", spans=[(12000, 13800)], qtype="fact",
#          source="llm-draft-edited")
# add_item("f19", question="Wie heißt Frankensteins Hund?", answer="Steht nicht im Buch.",
#          spans=[], qtype="unanswerable")

## 6. Smoke-Eval der neuen Items (Span-Hit/MRR mit der echten Suche)

In [ ]:
def span_hit(point_row: dict, item: dict) -> bool:
    if point_row.get("book_id") not in (None, item["book_id"]):
        return False
    return any(point_row["char_start"] < s["char_end"] and s["char_start"] < point_row["char_end"]
               for s in item["spans"])


def smoke_eval(items: list[dict], k: int = 8) -> pd.DataFrame:
    rows = []
    for item in items:
        if item["qtype"] == "unanswerable":
            continue
        cands = preview_search(item["question"], k=k)
        hits = [r + 1 for r, c in cands.iterrows() if span_hit(c.to_dict(), item)]
        rows.append({"id": item["id"], "qtype": item["qtype"],
                     "erster_hit_rank": hits[0] if hits else None,
                     "hit@1": bool(hits and hits[0] == 1), f"hit@{k}": bool(hits),
                     "rr": 1 / hits[0] if hits else 0.0})
    df = pd.DataFrame(rows)
    if len(df):
        print(f"Hit-Rate@1: {df['hit@1'].mean():.2f}   Hit-Rate@{k}: {df[f'hit@{k}'].mean():.2f}   "
              f"MRR: {df['rr'].mean():.3f}   (n={len(df)})")
    return df


smoke_eval(gold_new)

## 7. Export

Schreibt `eval/gold_frankenstein.jsonl`. Überschreibt nie; IDs müssen mit `f` beginnen, damit sie
nicht mit Dracula (`d…`, `n…`, `u…`) kollidieren.

In [ ]:
import os
from collections import Counter

SAVE = False

if SAVE:
    assert gold_new, "nichts zu exportieren"
    assert all(i["id"].startswith(ID_PREFIX) for i in gold_new), "IDs müssen mit 'f' beginnen"
    assert not os.path.exists(GOLD_OUT), f"{GOLD_OUT} existiert schon — erst umbenennen."
    with open(GOLD_OUT, "w", encoding="utf-8") as f:
        for item in gold_new:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"{len(gold_new)} Items → {GOLD_OUT}", Counter(i["qtype"] for i in gold_new))
else:
    print(f"SAVE=False — {len(gold_new)} Items im Speicher, nichts geschrieben.",
          Counter(i["qtype"] for i in gold_new))